In [1]:
import duckdb
from pathlib import Path
import plotly.express as px

from charting_helpers import LandRegistryChartingHelper
from data_wranglers import LandRegistryDataWrangler


In [2]:
LAND_REGISTRY_DATABASE_PATH = Path("../../databases/land_registry.db")

In [3]:
land_registry_data_wrangler = LandRegistryDataWrangler(LAND_REGISTRY_DATABASE_PATH)

In [4]:
db = duckdb.connect(LAND_REGISTRY_DATABASE_PATH, read_only=True)

# High level overview of data


In [5]:
land_registry_data_wrangler.get_min_max_dates()

(datetime.date(2023, 1, 1), datetime.date(2026, 3, 31))

In [6]:
land_registry_data_wrangler.get_property_types()

['Semi-Detached', 'Detached', 'Flat', 'Terraced']

In [7]:
db.query("FROM gold.median_price_by_month_and_property_type LIMIT 5").pl()

property_type,month_of_sale,median_price
str,datetime[μs],f64
"""Detached""",2023-01-01 00:00:00,430000.0
"""Flat""",2023-01-01 00:00:00,238000.0
"""Semi-Detached""",2023-01-01 00:00:00,260000.0
"""Terraced""",2023-01-01 00:00:00,212795.0
"""Detached""",2023-02-01 00:00:00,420000.0


In [8]:
LandRegistryChartingHelper.chart_median_price_by_month_and_property_type(db.query("FROM gold.median_price_by_month_and_property_type"))

## Generate Insights for Postcode

We can now use DuckDB as source to generate interesting insights for spcific slices of the data.

With plotly charting library to visualise the results.

In [9]:
POSTCODE = "NP19"

In [10]:
summary_statistics_for_postcode = land_registry_data_wrangler.generate_summary_statistics_for_postcode(
    postcode=POSTCODE,
    property_type=None,
    min_date=None,
    max_date=None,
)

In [11]:
summary_statistics_for_postcode

┌───────────────┬───────────┬───────────┬──────────────┬─────────────────┐
│ property_type │ max_price │ min_price │ median_price │ number_of_sales │
│    varchar    │   int64   │   int64   │    double    │      int64      │
├───────────────┼───────────┼───────────┼──────────────┼─────────────────┤
│ Detached      │    712500 │    140000 │     316000.0 │             391 │
│ Flat          │    217500 │     38500 │     127750.0 │             266 │
│ Semi-Detached │    430000 │     73500 │     240000.0 │             886 │
│ Terraced      │    420000 │     57500 │     170000.0 │            1376 │
└───────────────┴───────────┴───────────┴──────────────┴─────────────────┘

In [12]:
insights_for_postcode = land_registry_data_wrangler.generate_insights_for_postcode(
    postcode=POSTCODE,
    property_type=None,
    min_date=None,
    max_date=None,
)

In [13]:
insights_for_postcode

┌───────────────┬─────────────────────┬───────────┬───────────┬──────────────┬─────────────────┐
│ property_type │    month_of_sale    │ max_price │ min_price │ median_price │ number_of_sales │
│    varchar    │      timestamp      │   int64   │   int64   │    double    │      int64      │
├───────────────┼─────────────────────┼───────────┼───────────┼──────────────┼─────────────────┤
│ Detached      │ 2023-01-01 00:00:00 │    425995 │    275000 │     320000.0 │               7 │
│ Flat          │ 2023-01-01 00:00:00 │    175000 │     78000 │     129997.5 │              14 │
│ Semi-Detached │ 2023-01-01 00:00:00 │    379995 │    120000 │     235000.0 │              17 │
│ Terraced      │ 2023-01-01 00:00:00 │    274995 │    106000 │     168997.5 │              36 │
│ Detached      │ 2023-02-01 00:00:00 │    455000 │    278995 │     364997.5 │               4 │
│ Flat          │ 2023-02-01 00:00:00 │    175000 │     75000 │     130900.0 │              11 │
│ Semi-Detached │ 2023-02-01 0

In [14]:
LandRegistryChartingHelper.chart_median_price_by_month_and_property_type(insights_for_postcode)

In [15]:
LandRegistryChartingHelper.chart_number_of_sales_by_month_and_property_type(insights_for_postcode)

## Plot individual properties

In [16]:
individual_properties = land_registry_data_wrangler.get_all_matching_properties(
    postcode=POSTCODE,
    property_type=None,
    min_date=None,
    max_date=None,
)

In [19]:
LandRegistryChartingHelper.map_individual_property_sales(individual_properties)